In [ ]:
import os
import sys

os.environ["TQDM_DISABLE"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"

import tqdm.auto
import tqdm.std

tqdm.auto.tqdm = tqdm.std.tqdm
tqdm.auto.trange = tqdm.std.trange

import logging
from pathlib import Path

import pandas as pd

from drzavni_posao import (
    CharacterAssignerConfig,
    EmbedderConfig,
    MediaStore,
    MlflowTracker,
    PipelineConfig,
    Stage,
    TranscriberConfig,
    evaluate_run,
    run_pipeline,
)
from drzavni_posao.gallery import load_speaker_samples
from drzavni_posao.metrics import (
    assignment_scores,
    character_table,
    confusion_table,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)

logging.getLogger("py4j").setLevel(logging.WARNING)
logging.getLogger("mlflow.system_metrics").setLevel(logging.WARNING)

logger = logging.getLogger("drzavni_posao.pipeline")


In [ ]:
EXPERIMENT_NAME = (
    "/Users/luka.erceg@databricks.com/drzavni_posao/transcription"
)
AUDIO_PATHS = Path("/Volumes/luka_erceg/drzavni_posao/audios")
TRANSCRIPT_PATH = Path(
    "/Workspace/Users/luka.erceg@databricks.com/drzavni_posao/transcripts"
)
LABEL_DIR = Path(
    "/Workspace/Users/luka.erceg@databricks.com/drzavni_posao/labels"
)
GALLERY_PATH = Path(
    "/Workspace/Users/luka.erceg@databricks.com/drzavni_posao/labels/"
    "gallery_extended.json"
)
SOURCE_RUN_ID = "8bcaa3e6aa1b48849da6a9c86810e19f"
EVAL_CACHE = None  # a local directory keeps re-scoring off the wire

HF_TOKEN = dbutils.secrets.get(scope="luka-erceg", key="hf_token")

DEVICE = "cuda"
EMBEDDER_MODEL = "pyannote/wespeaker-voxceleb-resnet34-LM"

tracker = MlflowTracker(EXPERIMENT_NAME)


In [ ]:
SPEAKER_SAMPLES = load_speaker_samples(GALLERY_PATH)


In [ ]:
source = tracker._configuration(SOURCE_RUN_ID)

media = MediaStore(
    AUDIO_PATHS,
    TRANSCRIPT_PATH,
    **source["audio"],
)

character_assigner_config = CharacterAssignerConfig(
    min_similarity=0.1,
    embedder=EmbedderConfig(
        device=DEVICE,
        hf_token=HF_TOKEN,
        embed_model=EMBEDDER_MODEL,
        min_segment_seconds=0.01,
    ),
    speaker_samples=SPEAKER_SAMPLES,
)

config = PipelineConfig.from_media(
    media,
    transcription=TranscriberConfig(**source["transcription"]),
    character_assigner=character_assigner_config,
)

ALL_EPISODES = {episode.key for episode in media.episodes()}


In [ ]:
tracker.fork_run(
    SOURCE_RUN_ID,
    config.generated_name(),
    config,
    copy_until=Stage.TRANSCRIPTION,
)


In [ ]:
ASSIGNER_RUN_ID = run_pipeline(
    tracker,
    media,
    config,
    ALL_EPISODES,
    through=Stage.CHARACTER_ASSIGNER,
    tags={"experiment": "character-assigner", "gallery": "extended"},
)
ASSIGNER_RUN_ID


In [ ]:
scored = evaluate_run(tracker, ASSIGNER_RUN_ID, LABEL_DIR, EVAL_CACHE)
pd.DataFrame([assignment_scores(scored)]).T


In [ ]:
# Drill-down: which episode drags accuracy down, which characters it trades for
# each other, and the longest stretches it got wrong. A character with high
# recall but low precision is being over-assigned; time in the [anonymous]
# column is the assigner declining rather than confusing.
per_episode = pd.DataFrame(
    {
        key: assignment_scores(episode)
        for key, episode in scored.groupby("episode_key")
    }
).T.sort_values("accuracy")

display(per_episode.round(3))
display(confusion_table(scored).round(1))
display(character_table(scored).round(3))
scored[~scored["correct"]].sort_values("seconds", ascending=False).head(20)
